In [ ]:

# Import des librairies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
from google.colab import files
uploaded = files.upload("NYC.csv")

KeyboardInterrupt: 

In [ ]:
df = pd.read_csv("NYC.csv/NYC.csv")

In [ ]:
df["log_duration"] = np.log1p(df["trip_duration"])
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])
df["pickup_hour"] = df["pickup_datetime"].dt.hour
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])
df["pickup_month"] = df["pickup_datetime"].dt.month

# 3. Analyse Exploratoire (EDA)

## ·	Statistiques descriptives

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()

desc = df[num_cols].describe().T
desc["median"] = df[num_cols].median()
desc["skewness"] = df[num_cols].skew()
desc["kurtosis"] = df[num_cols].kurtosis()
desc["missing"] = df[num_cols].isna().sum()
desc["missing_%"] = (df[num_cols].isna().mean() * 100).round(2)

display(desc.style
    .background_gradient(cmap="Blues", subset=["mean","std"])
    .background_gradient(cmap="Reds",  subset=["missing_%"])
    .format(precision=3))


## ·	 Visualisation

###	Histogrammes

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371

    lat1, lon1, lat2, lon2 = map(
        np.radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

df["distance_km"] = haversine(
    df["pickup_latitude"],
    df["pickup_longitude"],
    df["dropoff_latitude"],
    df["dropoff_longitude"]
)

In [ ]:
df["log_duration"] = np.log1p(df["trip_duration"])

plot_cols = [
    "trip_duration",
    "log_duration",
    "distance_km",
    "passenger_count",
    "pickup_hour"
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

colors = sns.color_palette("viridis", len(plot_cols))

for i, col in enumerate(plot_cols):
    data = df[col].dropna()

    axes[i].hist(
        data,
        bins=60,
        color=colors[i],
        edgecolor="white",
        linewidth=0.4
    )

    axes[i].set_title(col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Fréquence")

    # Médiane
    med = data.median()

    axes[i].axvline(
        med,
        color="crimson",
        linestyle="--",
        linewidth=1.5,
        label=f"Médiane = {med:.1f}"
    )

    axes[i].legend(fontsize=8)

# cacher le subplot vide
axes[-1].set_visible(False)

fig.suptitle(
    "Distribution des variables numériques — NYC Taxi",
    fontsize=15,
    fontweight="bold",
    y=1.01
)

plt.tight_layout()
plt.show()

###	Boxplots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1) trip_duration brut
sns.boxplot(y=df["trip_duration"], ax=axes[0],
            color="#3498db", flierprops=dict(marker=".", markersize=2, alpha=0.3))
axes[0].set_title("trip_duration (brut)")
axes[0].set_ylabel("Secondes")

# 2) log_duration
sns.boxplot(y=df["log_duration"], ax=axes[1],
            color="#2ecc71", flierprops=dict(marker=".", markersize=2, alpha=0.3))
axes[1].set_title("log(trip_duration + 1)")
axes[1].set_ylabel("Log-secondes")

# 3) duration par vendor_id
sns.boxplot(x="vendor_id", y="log_duration", data=df, ax=axes[2],
            palette="Set2", flierprops=dict(marker=".", markersize=2, alpha=0.3))
axes[2].set_title("log_duration par vendor_id")
axes[2].set_xlabel("Vendor ID")

fig.suptitle("Boxplots — NYC Taxi Trip Duration", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


###Heatmap (corrélation)

In [ ]:
corr_cols = ["trip_duration", "log_duration", "distance_km",
             "passenger_count", "pickup_hour", "pickup_month",
             "pickup_latitude", "pickup_longitude",
             "dropoff_latitude", "dropoff_longitude"]

corr_matrix = df[corr_cols].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"shrink": 0.8, "label": "Corrélation de Pearson"},
    ax=ax
)
ax.set_title("Matrice de corrélation — NYC Taxi Trip Duration",
             fontsize=14, fontweight="bold", pad=15)
plt.xticks(rotation=40, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("\n📌 Top corrélations avec trip_duration :")
top_corr = corr_matrix["trip_duration"].drop("trip_duration").sort_values(key=abs, ascending=False)
print(top_corr.map("{:.3f}".format).to_string())


##· Détection

###	Valeurs manquantes

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_df = pd.DataFrame({"Manquants": missing, "% total": missing_pct})
missing_df = missing_df[missing_df["Manquants"] > 0]

if missing_df.empty:
    print(" Aucune valeur manquante détectée.")
else:
    display(missing_df.style.background_gradient(cmap="OrRd", subset=["% total"]))

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.barh(missing_df.index, missing_df["% total"], color="#e74c3c", edgecolor="white")
    ax.set_xlabel("% de valeurs manquantes")
    ax.set_title("Colonnes avec valeurs manquantes")
    for i, v in enumerate(missing_df["% total"]):
        ax.text(v + 0.02, i, f"{v:.2f}%", va="center", fontsize=10)
    plt.tight_layout()
    plt.show()


### Outliers

In [ ]:
def detect_outliers_iqr(series, factor=1.5):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - factor * IQR, Q3 + factor * IQR
    mask = (series < lower) | (series > upper)
    return mask, lower, upper

outlier_cols = ["trip_duration", "distance_km", "passenger_count"]
results = []

for col in outlier_cols:
    mask, lo, hi = detect_outliers_iqr(df[col].dropna())
    n_out = mask.sum()
    pct   = 100 * n_out / len(mask)
    results.append({"Colonne": col,
                    "Borne inf (IQR)": round(lo, 2),
                    "Borne sup (IQR)": round(hi, 2),
                    "# Outliers"     : n_out,
                    "% Outliers"     : round(pct, 2)})

outlier_df = pd.DataFrame(results)
display(outlier_df.style
    .background_gradient(cmap="YlOrRd", subset=["% Outliers"])
    .format({"% Outliers": "{:.2f}%"}))

# Z-score sur trip_duration
z_scores = np.abs(stats.zscore(df["trip_duration"].dropna()))
print(f"\n📌 Outliers Z-score > 3 sur trip_duration : {(z_scores > 3).sum():,} "
      f"({100*(z_scores > 3).mean():.2f}%)")


###Déséquilibre des classes

In [ ]:
cat_cols = ["vendor_id", "store_and_fwd_flag", "passenger_count"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(cat_cols):
    vc = df[col].value_counts()
    pct = (vc / len(df) * 100).round(1)
    bars = axes[i].bar(vc.index.astype(str), vc.values,
                       color=sns.color_palette("pastel"), edgecolor="grey")
    axes[i].set_title(f"Distribution : {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Effectif")

    for bar, p in zip(bars, pct.values):
        axes[i].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 50,
                     f"{p}%", ha="center", va="bottom", fontsize=9)

    # Ratio déséquilibre
    ratio = vc.max() / vc.min()
    axes[i].set_title(f"{col}  (ratio max/min = {ratio:.1f}×)")

fig.suptitle("Déséquilibre des classes — NYC Taxi", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n📌 Résumé store_and_fwd_flag (fort déséquilibre) :")
print(df["store_and_fwd_flag"].value_counts(normalize=True).map("{:.2%}".format))





**4.1 Vérifier les valeurs manquantes**

In [ ]:
df.isnull().sum()

In [ ]:
df.fillna(df.median(numeric_only=True), inplace=True)

**4.2 Vérifier les doublons**

In [ ]:
print("Doublons :", df.duplicated().sum())

In [ ]:
df = df.drop_duplicates()

**4.3 Encoder les variables catégorielles**

In [ ]:
df["store_and_fwd_flag"] = df["store_and_fwd_flag"].map({
    "N": 0,
    "Y": 1
})

 **4.4 Supprimer les colonnes inutiles**

In [ ]:
df = df.drop(["id"], axis=1)

**4.5 Gérer les outliers**

In [ ]:
Q1 = df["trip_duration"].quantile(0.25)
Q3 = df["trip_duration"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df = df[(df["trip_duration"] >= lower) &
        (df["trip_duration"] <= upper)]

**4.6 Standardisation**

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [ ]:
num_cols = [
    "distance_km",
    "passenger_count",
    "pickup_hour",
    "pickup_month"
]

In [ ]:
df[num_cols] = scaler.fit_transform(df[num_cols])

**4.7 Vérification finale**

In [ ]:
df.info()
df.head()

# 5  Feature Engineering

In [ ]:



from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


In [ ]:
# Vérification
required_cols = [
    "trip_duration", "distance_km", "passenger_count",
    "pickup_hour", "pickup_month",
    "pickup_latitude", "pickup_longitude",
    "dropoff_latitude", "dropoff_longitude"
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Colonnes manquantes dans df : {missing_cols}")

print(f" DataFrame prêt : {df.shape}")

#  5.1 Création de nouvelles variables

In [ ]:
df_fe = df.copy()

# 5.1.1 Interaction distance × passagers
df_fe["dist_x_passengers"] = df_fe["distance_km"] * df_fe["passenger_count"]

# 5.1.2 Heures de pointe (heure réelle 0-23)
df_fe["is_rush_hour"] = (
    ((df_fe["pickup_hour"] >= 7) & (df_fe["pickup_hour"] <= 10)) |
    ((df_fe["pickup_hour"] >= 16) & (df_fe["pickup_hour"] <= 19))
).astype(int)

# 5.1.3 Période de la journée
# nuit [0-5], matin [6-11], après-midi [12-17], soir [18-23]
bins = [-1, 5, 11, 17, 23]
labels = ["night", "morning", "afternoon", "evening"]
df_fe["pickup_period"] = pd.cut(df_fe["pickup_hour"], bins=bins, labels=labels)
df_fe = pd.get_dummies(df_fe, columns=["pickup_period"], drop_first=True)

# 5.1.4 Log-distance
df_fe["log_distance"] = np.log1p(df_fe["distance_km"].clip(lower=0))

# 5.1.5 Distance² pour capturer non-linéarités
df_fe["distance_km_sq"] = df_fe["distance_km"] ** 2

# 5.1.6 Features géographiques robustes
df_fe["lat_diff"] = (df_fe["dropoff_latitude"] - df_fe["pickup_latitude"]).abs()
df_fe["lon_diff"] = (df_fe["dropoff_longitude"] - df_fe["pickup_longitude"]).abs()
df_fe["manhattan_dist_proxy"] = df_fe["lat_diff"] + df_fe["lon_diff"]

new_features = [
    "dist_x_passengers", "is_rush_hour", "log_distance", "distance_km_sq",
    "lat_diff", "lon_diff", "manhattan_dist_proxy"
]
new_features += [c for c in df_fe.columns if c.startswith("pickup_period_")]

print("\n Nouvelles features créées :")
print(new_features)

display(df_fe[new_features].describe(include="all").T)

# 5.2  Sélection des features importantes

In [ ]:
TARGET = "trip_duration"

# Features numériques uniquement (RF + Pearson)
candidate_features = [
    c for c in df_fe.select_dtypes(include=[np.number]).columns
    if c != TARGET
]

# Nettoyage NA pour le bloc sélection
X_all = df_fe[candidate_features]
y_all = df_fe[TARGET]
valid_idx = X_all.dropna().index.intersection(y_all.dropna().index)

X = X_all.loc[valid_idx]
y = y_all.loc[valid_idx]

print(f"\n Shape pour sélection : X={X.shape}, y={y.shape}")

# 5.2.1 Corrélation de Pearson absolue
pearson_corr = X.corrwith(y).abs().sort_values(ascending=False)

plt.figure(figsize=(12, 5))
pearson_corr.plot(kind="bar", color=sns.color_palette("viridis", len(pearson_corr)))
plt.title("Corrélation |Pearson| avec trip_duration", fontweight="bold")
plt.ylabel("|Corrélation|")
plt.axhline(0.10, color="red", linestyle="--", label="Seuil 0.10")
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

print("\n Corrélations triées :")
print(pearson_corr.map("{:.3f}".format).to_string())

# 5.2.2 Importance via Random Forest
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestRegressor(
    n_estimators=150,
    max_depth=12,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(12, 5))
importances.plot(kind="bar", color=sns.color_palette("rocket", len(importances)))
plt.title("Importance des features — Random Forest", fontweight="bold")
plt.ylabel("Importance")
plt.axhline(0.02, color="red", linestyle="--", label="Seuil 2%")
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

print("\n Importances RF triées :")
print(importances.map("{:.4f}".format).to_string())

# 5.2.3 Sélection finale
selected_rf = importances[importances > 0.02].index.tolist()
selected_pearson = pearson_corr[pearson_corr > 0.10].index.tolist()
selected_features = sorted(list(set(selected_rf) | set(selected_pearson)))

print(f"\n Features sélectionnées ({len(selected_features)}):")
for f in selected_features:
    print(" -", f)

df_selected = df_fe[selected_features + [TARGET]].copy()
print(f"\n Shape df_selected : {df_selected.shape}")

#  5.3 PCA (optionnelle)

In [ ]:
X_sel = df_selected.drop(columns=[TARGET]).dropna()
y_sel = df_selected.loc[X_sel.index, TARGET]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sel)

# 5.3.1 Courbe variance expliquée
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(cumvar)+1), cumvar, marker="o", color="#2980b9")
plt.axhline(0.90, color="red", linestyle="--", label="90%")
plt.axhline(0.95, color="orange", linestyle="--", label="95%")
plt.xlabel("Nombre de composantes")
plt.ylabel("Variance expliquée cumulée")
plt.title("PCA — Variance expliquée", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()

n_90 = np.argmax(cumvar >= 0.90) + 1
n_95 = np.argmax(cumvar >= 0.95) + 1
print(f" Composantes pour 90%: {n_90}")
print(f" Composantes pour 95%: {n_95}")

# 5.3.2 Application PCA à 95%
pca = PCA(n_components=n_95, random_state=42)
X_pca = pca.fit_transform(X_scaled)
pca_cols = [f"PC{i+1}" for i in range(n_95)]

df_pca = pd.DataFrame(X_pca, columns=pca_cols, index=X_sel.index)
df_pca[TARGET] = y_sel.values

print(f"\n PCA appliquée: {X_sel.shape[1]} -> {n_95} composantes")
print(f"   Variance conservée: {cumvar[n_95-1]*100:.1f}%")
print(f" Shape df_pca : {df_pca.shape}")

# 5.3.3 Loadings
loadings = pd.DataFrame(
    pca.components_.T,
    index=X_sel.columns,
    columns=pca_cols
)

plt.figure(figsize=(12, 6))
sns.heatmap(
    loadings.iloc[:, :min(5, n_95)],
    annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5
)
plt.title("Loadings PCA (5 premières composantes)", fontweight="bold")
plt.tight_layout()
plt.show()

if "PC1" in loadings.columns:
    print("\n Contributions dans PC1:")
    print(loadings["PC1"].sort_values(key=np.abs, ascending=False).map("{:.3f}".format).to_string())


In [ ]:
n_features_original = X_sel.shape[1]
reduction_ratio = (n_features_original - n_95) / n_features_original * 100

print("\n" + "="*60)
print("RÉSUMÉ — Étape 5 : Feature Engineering")
print("="*60)
print(f"Features candidates       : {len(candidate_features)}")
print(f"Nouvelles features créées : {len(new_features)}")
print(f"Features sélectionnées    : {len(selected_features)}")
print(f"PCA composantes (95%)     : {n_95}")
print(f"Réduction dimensionnelle  : {reduction_ratio:.1f}%")
print("="*60)

# Par défaut, garder interprétabilité
if reduction_ratio < 20:
    print("\n PCA réduit peu la dimension (<20%).")
    print(" Recommandation: utiliser df_selected pour la modélisation.")
    df_final = df_selected.copy()
else:
    print("\n PCA utile (réduction significative).")
    print(" Recommandation: tester df_selected et df_pca, comparer performances.")
    df_final = df_selected.copy()  # interprétable par défaut

print(f"\n df_final : {df_final.shape}")
display(df_final.head())

6. SPLIT DES DONNÉES

In [ ]:
TARGET = "trip_duration"

X = df_final.drop(columns=[TARGET])
y = df_final[TARGET]
print(f"Shape X : {X.shape}")
print(f"Shape y : {y.shape}")



# 6.2  Split Train / Test  (80 % / 20 %)

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print(f"\nAprès Train/Test split :")
print(f"  X_train_full : {X_train_full.shape}")
print(f"  X_test       : {X_test.shape}")

# 6.3  Split Train / Validation  (80 % / 20 % du train)
#      → 64 % / 16 % / 20 % du total

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.20,
    random_state=42
)

print(f"\nAprès Train/Val split :")
print(f"  X_train : {X_train.shape}")
print(f"  X_val   : {X_val.shape}")
print(f"  X_test  : {X_test.shape}")


# 6.4  Vérification des proportions

In [ ]:
total = len(X)
print(f"\nProportions réelles :")
print(f"  Train      : {len(X_train)/total*100:.1f} %")
print(f"  Validation : {len(X_val)/total*100:.1f}  %")
print(f"  Test       : {len(X_test)/total*100:.1f}  %")

# 6.5  Vérification : pas de fuite de données

In [ ]:
train_idx = set(X_train.index)
val_idx   = set(X_val.index)
test_idx  = set(X_test.index)

assert train_idx.isdisjoint(val_idx),  "⚠️  Fuite train ↔ val !"
assert train_idx.isdisjoint(test_idx), "⚠️  Fuite train ↔ test !"
assert val_idx.isdisjoint(test_idx),   "⚠️  Fuite val ↔ test !"

print("\n✅ Aucune fuite de données détectée.")

In [ ]:
# 6.6  Résumé statistique (vérifier cohérence des splits)

In [ ]:
summary = pd.DataFrame({
    "Train"      : y_train.describe(),
    "Validation" : y_val.describe(),
    "Test"       : y_test.describe(),
})
print("\nStatistiques de y par split :")
display(summary.style.background_gradient(cmap="Blues", axis=1))



# 7. MODÉLISATION

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVR, SVC
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# PART A — RÉGRESSION
# Prédire trip_duration (en secondes) directement

In [ ]:

print("=" * 60)
print("PART A — RÉGRESSION")
print("=" * 60)

 # A.1  Modèles

In [ ]:
regression_models = {
    "Linear Regression"    : LinearRegression(),
    "Random Forest Regressor" : RandomForestRegressor(
                                    n_estimators=100,
                                    max_depth=10,
                                    random_state=42,
                                    n_jobs=-1
                                ),
}

# A.2  Entraînement + Évaluation sur Val

In [ ]:
reg_results = []

for name, model in regression_models.items():
    # Entraînement
    model.fit(X_train, y_train)

    # Prédictions
    y_pred_val   = model.predict(X_val)
    y_pred_train = model.predict(X_train)

    # Métriques
    mae  = mean_absolute_error(y_val, y_pred_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
    r2   = r2_score(y_val, y_pred_val)
    r2_train = r2_score(y_train, y_pred_train)

    reg_results.append({
        "Modèle"       : name,
        "MAE (val)"    : round(mae, 2),
        "RMSE (val)"   : round(rmse, 2),
        "R² (val)"     : round(r2, 4),
        "R² (train)"   : round(r2_train, 4),
        "Overfitting?" : "⚠️ Oui" if (r2_train - r2) > 0.10 else "✅ Non"
    })

    print(f"\n{name}")
    print(f"  MAE   = {mae:,.2f} sec")
    print(f"  RMSE  = {rmse:,.2f} sec")
    print(f"  R² val   = {r2:.4f}")
    print(f"  R² train = {r2_train:.4f}")

reg_df = pd.DataFrame(reg_results)
display(reg_df.style.background_gradient(cmap="Greens", subset=["R² (val)"]))


# A.3  Visualisation : Prédit vs Réel

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, model) in zip(axes, regression_models.items()):
    y_pred = model.predict(X_val)
    ax.scatter(y_val, y_pred, alpha=0.3, s=8, color="#3498db")
    lims = [min(y_val.min(), y_pred.min()),
            max(y_val.max(), y_pred.max())]
    ax.plot(lims, lims, "r--", linewidth=1.5, label="Parfait")
    ax.set_xlabel("Durée réelle (sec)")
    ax.set_ylabel("Durée prédite (sec)")
    ax.set_title(name)
    ax.legend()

fig.suptitle("Régression — Prédit vs Réel (Validation)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# PART B — CLASSIFICATION
# Convertir trip_duration en 3 classes : Court / Moyen / Long

In [ ]:
print("\n" + "=" * 60)
print("PART B — CLASSIFICATION")
print("=" * 60)

# B.1  Créer la variable cible catégorielle

In [ ]:
def make_duration_class(y_series):
    """Bins basés sur les terciles de y_train."""
    q33 = y_series.quantile(0.33)
    q66 = y_series.quantile(0.66)
    return pd.cut(
        y_series,
        bins=[-np.inf, q33, q66, np.inf],
        labels=["Court", "Moyen", "Long"]
    )

# Calculer les seuils sur y_train uniquement (pas de fuite)

In [ ]:
q33 = y_train.quantile(0.33)
q66 = y_train.quantile(0.66)

print(f"Seuils (terciles de y_train) :")
print(f"  Court  < {q33:.0f} sec  ({q33/60:.1f} min)")
print(f"  Moyen  [{q33:.0f} – {q66:.0f}] sec")
print(f"  Long   > {q66:.0f} sec  ({q66/60:.1f} min)")

y_train_cls = pd.cut(y_train, bins=[-np.inf, q33, q66, np.inf],
                     labels=["Court", "Moyen", "Long"])
y_val_cls   = pd.cut(y_val,   bins=[-np.inf, q33, q66, np.inf],
                     labels=["Court", "Moyen", "Long"])

print(f"\nDistribution des classes (train) :")
print(y_train_cls.value_counts().sort_index())

# B.2  Normaliser X pour LR et SVM

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)

# B.3  Modèles de classification

In [ ]:
clf_models = {
    "Logistic Regression" : (LogisticRegression(
                                max_iter=500,
                                random_state=42,
                                n_jobs=-1
                             ), True),   # True = utiliser X scalé
    "Random Forest"       : (RandomForestClassifier(
                                n_estimators=100,
                                max_depth=10,
                                random_state=42,
                                n_jobs=-1
                             ), False),
    "SVM"                 : (SVC(
                                kernel="rbf",
                                C=1.0,
                                random_state=42
                             ), True),
}

# B.4  Entraînement + Évaluation

In [ ]:
clf_results = []

for name, (model, use_scaled) in clf_models.items():
    Xtr = X_train_sc if use_scaled else X_train
    Xvl = X_val_sc   if use_scaled else X_val

    model.fit(Xtr, y_train_cls)
    y_pred = model.predict(Xvl)

    report = classification_report(y_val_cls, y_pred, output_dict=True)
    acc    = report["accuracy"]

    clf_results.append({
        "Modèle"        : name,
        "Accuracy"      : round(acc, 4),
        "F1 Court"      : round(report["Court"]["f1-score"], 4),
        "F1 Moyen"      : round(report["Moyen"]["f1-score"], 4),
        "F1 Long"       : round(report["Long"]["f1-score"],  4),
    })

    print(f"\n{'─'*40}")
    print(f"{name}")
    print(f"{'─'*40}")
    print(classification_report(y_val_cls, y_pred,
                                target_names=["Court","Moyen","Long"]))

clf_df = pd.DataFrame(clf_results)
display(clf_df.style.background_gradient(cmap="Blues", subset=["Accuracy"]))

# B.5  Matrices de confusion

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, (model, use_scaled)) in zip(axes, clf_models.items()):
    Xvl    = X_val_sc if use_scaled else X_val
    y_pred = model.predict(Xvl)
    cm     = confusion_matrix(y_val_cls, y_pred,
                              labels=["Court","Moyen","Long"])
    disp   = ConfusionMatrixDisplay(cm, display_labels=["Court","Moyen","Long"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name)

fig.suptitle("Matrices de confusion — Classification (Validation)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# B.6  Résumé final

In [ ]:
print("\n" + "=" * 60)
print("RÉSUMÉ — Section 7")
print("=" * 60)
print("\nRégression :")
display(reg_df[["Modèle","MAE (val)","RMSE (val)","R² (val)","Overfitting?"]])

print("\nClassification :")
display(clf_df.sort_values("Accuracy", ascending=False))

best_reg = reg_df.loc[reg_df["R² (val)"].idxmax(), "Modèle"]
best_clf = clf_df.loc[clf_df["Accuracy"].idxmax(), "Modèle"]
print(f"\n🏆 Meilleur modèle régression   : {best_reg}")
print(f"🏆 Meilleur modèle classification: {best_clf}")

# 8

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

## 8.1 Régression

### 8.1.1 Métriques — MSE · RMSE · R²


In [ ]:
reg_eval_rows = []

for name, model in regression_models.items():
    y_pred_test = model.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred_test)
    mse  = mean_squared_error(y_test, y_pred_test)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_test, y_pred_test)

    reg_eval_rows.append({
        "Modele" : name,
        "MAE"    : round(mae,  2),
        "MSE"    : round(mse,  2),
        "RMSE"   : round(rmse, 2),
        "R²"     : round(r2,   4),
    })

    print(f"\n{'═'*50}")
    print(f"  {name}")
    print(f"{'═'*50}")
    print(f"  MAE  = {mae:>12,.2f} sec")
    print(f"  MSE  = {mse:>12,.2f} sec²")
    print(f"  RMSE = {rmse:>12,.2f} sec  ({rmse/60:.1f} min)")
    print(f"  R²   = {r2:>12.4f}")

df_reg_eval = pd.DataFrame(reg_eval_rows)
display(df_reg_eval.style
    .background_gradient(cmap="Greens", subset=["R²"])
    .background_gradient(cmap="Reds_r", subset=["RMSE", "MSE"])
    .format({"MAE": "{:,.2f}", "MSE": "{:,.2f}", "RMSE": "{:,.2f}", "R²": "{:.4f}"}))

# Meilleur modèle régression
best_reg_name  = df_reg_eval.loc[df_reg_eval["R²"].idxmax(), "Modele"]
best_reg_model = regression_models[best_reg_name]
y_pred_test    = best_reg_model.predict(X_test)
print(f"\n Meilleur modèle régression : {best_reg_name}")

### 8.1.2 Réel vs Prédit

In [ ]:
plt.figure(figsize=(7, 6))
sample_idx = np.random.choice(len(y_test), size=min(5000, len(y_test)), replace=False)
plt.scatter(y_test.iloc[sample_idx], y_pred_test[sample_idx],
            alpha=0.25, s=12, color="#3498db")
lims = [min(y_test.min(), y_pred_test.min()),
        max(y_test.max(), y_pred_test.max())]
plt.plot(lims, lims, 'r--', linewidth=2, label="Parfait")
plt.xlabel("Valeurs réelles (trip_duration)")
plt.ylabel("Prédictions")
plt.title(f"Réel vs Prédit — {best_reg_name}", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()


### 8.1.3 Distribution des résidus

In [ ]:
residuals = y_test.values - y_pred_test

plt.figure(figsize=(8, 4))
sns.histplot(residuals, bins=80, kde=True, color="#9b59b6")
plt.axvline(0, color="red", linestyle="--", linewidth=1.5, label="Résidu = 0")
plt.axvline(np.mean(residuals), color="orange", linestyle="-.",
            linewidth=1.5, label=f"Moyenne = {np.mean(residuals):.1f}")
plt.title(f"Distribution des résidus — {best_reg_name}", fontweight="bold")
plt.xlabel("Résidu (y_true − y_pred)")
plt.legend()
plt.tight_layout()
plt.show()


---
## 8.2 Classification

Cible binaire : **long_trip = 1** si `trip_duration > médiane`, sinon 0.

### 8.2.1 Préparation des données

In [ ]:
TARGET = "trip_duration"

df_cls = df_final.copy()
threshold = df_cls[TARGET].median()
df_cls["long_trip"] = (df_cls[TARGET] > threshold).astype(int)

Xc = df_cls.drop(columns=[TARGET, "long_trip"])
yc = df_cls["long_trip"]

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.20, random_state=42, stratify=yc
)

print(f"Seuil (médiane) : {threshold:.0f} sec  ({threshold/60:.1f} min)")
print(f"Xc_train : {Xc_train.shape}  |  Xc_test : {Xc_test.shape}")
print(f"\nDistribution classes test :")
print(yc_test.value_counts().rename({0: 'Court (0)', 1: 'Long (1)'}))


### 8.2.2 Entraînement . Pipeline sklearn

> Ajouter une citation



In [ ]:
models_cls = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  LogisticRegression(max_iter=2000, random_state=42))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
    ),
    "SVM (RBF)": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  SVC(kernel="rbf", C=10, gamma="scale", random_state=42))
    ])
}

cls_rows = []

for name, model in models_cls.items():
    model.fit(Xc_train, yc_train)
    yc_pred = model.predict(Xc_test)

    cls_rows.append({
        "Modèle"    : name,
        "Accuracy"  : round(accuracy_score(yc_test,  yc_pred), 4),
        "Precision" : round(precision_score(yc_test, yc_pred), 4),
        "Recall"    : round(recall_score(yc_test,    yc_pred), 4),
        "F1-score"  : round(f1_score(yc_test,        yc_pred), 4),
    })

df_cls_results = pd.DataFrame(cls_rows).sort_values("F1-score", ascending=False)

display(df_cls_results.style
    .background_gradient(cmap="Blues", subset=["Accuracy", "F1-score"])
    .format(precision=4))

best_cls_name  = df_cls_results.iloc[0]["Modèle"]
best_cls_model = models_cls[best_cls_name]
print(f"\n Meilleur modèle classification : {best_cls_name}")

### 8.2.3 Classification Report

In [ ]:
yc_pred_best = best_cls_model.predict(Xc_test)
print(f"Classification Report — {best_cls_name}\n")
print(classification_report(yc_test, yc_pred_best,
                             target_names=["Court (0)", "Long (1)"]))

### 8.2.4 Matrice de confusion

In [ ]:
cm      = confusion_matrix(yc_test, yc_pred_best)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, shrink=0.85, label="Proportion")

labels = ["Court (0)", "Long (1)"]
ax.set_xticks([0,1]); ax.set_xticklabels(labels)
ax.set_yticks([0,1]); ax.set_yticklabels(labels)

thresh = cm_norm.max() / 2.0
for i in range(2):
    for j in range(2):
        ax.text(j, i,
                f"{cm[i,j]}\n({cm_norm[i,j]*100:.1f}%)",
                ha="center", va="center",
                color="white" if cm_norm[i,j] > thresh else "black",
                fontsize=11)

ax.set_xlabel("Classe prédite")
ax.set_ylabel("Classe réelle")
ax.set_title(f"Matrice de confusion — {best_cls_name}", fontweight="bold")
plt.tight_layout()
plt.show()


# **9.Optimisation**

**9.1 Cross-Validation**

In [ ]:
from sklearn.model_selection import cross_val_score

# Pour le meilleur modèle de régression (ex: Random Forest)
scores_reg = cross_val_score(best_reg_model, X_train, y_train,
                              cv=5, scoring='r2', n_jobs=-1)
print(f"R² moyen (CV 5-fold) : {scores_reg.mean():.4f} ± {scores_reg.std():.4f}")

# Pour le meilleur modèle de classification
scores_clf = cross_val_score(best_cls_model, Xc_train, yc_train,
                              cv=5, scoring='f1', n_jobs=-1)
print(f"F1 moyen (CV 5-fold) : {scores_clf.mean():.4f} ± {scores_clf.std():.4f}")

**9.2 Grid search/ Random search**

**Pour Random Forest(Regression):**

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5]
\}

grid_rf = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid_rf,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)
grid_rf.fit(X_train, y_train)
print(f"Meilleurs paramètres : {grid_rf.best_params_\}")
print(f"Meilleur R² (CV)     : {grid_rf.best_score_:.4f\}")$0

**Pour Random Forest(Classification)**

In [ ]:
param_grid_clf = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

grid_clf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid_clf,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)
grid_clf.fit(Xc_train, yc_train)
print(f"Meilleurs paramètres : {grid_clf.best_params_}")
print(f"Meilleur F1 (CV)     : {grid_clf.best_score_:.4f}")

**9.3 — Ré-évaluation avec les modèles optimisés**

Comparez les performances avant/après optimisation :

In [ ]:
# Régression optimisée
y_pred_opt = grid_rf.best_estimator_.predict(X_test)
r2_opt = r2_score(y_test, y_pred_opt)
rmse_opt = np.sqrt(mean_squared_error(y_test, y_pred_opt))
print(f"Après optimisation → R² : {r2_opt:.4f} | RMSE : {rmse_opt:.2f}")

# Classification optimisée
yc_pred_opt = grid_clf.best_estimator_.predict(Xc_test)
print(classification_report(yc_test, yc_pred_opt,
                             target_names=["Court (0)", "Long (1)"]))

**9.4 — Amélioration desfeatures**

In [ ]:
# Vitesse moyenne estimée
df["speed_kmh"] = df["distance_km"] / (df["trip_duration"] / 3600)

# Moment de la journée (matin, après-midi, soir, nuit)
df["time_of_day"] = pd.cut(df["pickup_hour"],
                            bins=[0, 6, 12, 18, 24],
                            labels=["Nuit", "Matin", "Après-midi", "Soir"],
                            right=False)

# **10. Déploiement**


**10.1 Sauvegarde du modèle**

In [ ]:
print(globals().keys())

In [ ]:
best_reg_model = _get_best_reg_model()
best_cls_model = _get_best_cls_model()

In [ ]:
import os
import joblib

# Créer le dossier
os.makedirs("model_artifacts", exist_ok=True)

# Sauvegarder les meilleurs modèles
joblib.dump(best_reg_model, "model_artifacts/best_reg_model.joblib")
joblib.dump(best_cls_model, "model_artifacts/best_cls_model.joblib")

# Sauvegarder les noms des variables
feature_names = list(X_train.columns)
joblib.dump(feature_names, "model_artifacts/feature_names.joblib")

print("Modèles sauvegardés avec succès :")
for f in os.listdir("model_artifacts"):
    size = os.path.getsize(os.path.join("model_artifacts", f)) / 1024
    print(f"{f:40s} {size:.1f} Ko")

**10.2 API REST avec Flask**

In [ ]:
flask_code = '''
# app_flask.py
from flask import Flask, request, jsonify
import joblib
import pandas as pd
import numpy as np

app = Flask(__name__)

# Chargement des artefacts
reg_model    = joblib.load("model_artifacts/best_reg_model.joblib")
cls_model    = joblib.load("model_artifacts/best_cls_model.joblib")
feature_cols = joblib.load("model_artifacts/feature_names.joblib")

def parse_input(data: dict) -> pd.DataFrame:
    """Convertit le JSON entrant en DataFrame avec les bonnes colonnes."""
    df_in = pd.DataFrame([data])
    for col in feature_cols:
        if col not in df_in.columns:
            df_in[col] = 0          # valeur par défaut
    return df_in[feature_cols]

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"}), 200

@app.route("/predict/duration", methods=["POST"])
def predict_duration():
    try:
        data   = request.get_json(force=True)
        X      = parse_input(data)
        pred   = reg_model.predict(X)[0]
        return jsonify({
            "trip_duration_sec" : round(float(pred), 1),
            "trip_duration_min" : round(float(pred) / 60, 2)
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 400

@app.route("/predict/type", methods=["POST"])
def predict_type():
    try:
        data   = request.get_json(force=True)
        X      = parse_input(data)
        label  = int(cls_model.predict(X)[0])
        proba  = cls_model.predict_proba(X)[0].tolist() \
                 if hasattr(cls_model, "predict_proba") else None
        return jsonify({
            "long_trip"    : bool(label),
            "label"        : label,
            "probabilities": proba
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 400

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)
'''

with open('app_flask.py', 'w') as f:
    f.write(flask_code.strip())

print('app_flask.py généré.')
print()
print('Pour lancer l\'API :')
print('  pip install flask')
print('  python app_flask.py')
print()
print('Exemple de requête curl :')
print('  curl -X POST http://localhost:5000/predict/duration \\')
print('       -H "Content-Type: application/json" \\')
print('       -d \'{"pickup_hour":8, "distance_km":3.5, "passenger_count":1}\'')

**10.3 API REST avec FastAPI**

In [ ]:
fastapi_code = '''
# app_fastapi.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional, List
import joblib
import pandas as pd
import numpy as np

app = FastAPI(
    title       = "NYC Taxi Trip Predictor",
    description = "Prédit la durée et le type d\'un trajet en taxi new-yorkais",
    version     = "1.0.0"
)

# Chargement des artefacts
reg_model    = joblib.load("model_artifacts/best_reg_model.joblib")
cls_model    = joblib.load("model_artifacts/best_cls_model.joblib")
feature_cols = joblib.load("model_artifacts/feature_names.joblib")

# ── Schémas Pydantic ────────────────────────────────────────────────────────
class TripFeatures(BaseModel):
    pickup_hour       : int   = Field(..., ge=0, le=23, description="Heure du départ (0-23)")
    distance_km       : float = Field(..., gt=0,        description="Distance estimée (km)")
    passenger_count   : int   = Field(1,  ge=1, le=6,  description="Nombre de passagers")
    pickup_longitude  : Optional[float] = None
    pickup_latitude   : Optional[float] = None
    dropoff_longitude : Optional[float] = None
    dropoff_latitude  : Optional[float] = None

class DurationResponse(BaseModel):
    trip_duration_sec : float
    trip_duration_min : float

class TypeResponse(BaseModel):
    long_trip     : bool
    label         : int
    probabilities : Optional[List[float]] = None

# ── Helpers ─────────────────────────────────────────────────────────────────
def to_dataframe(features: TripFeatures) -> pd.DataFrame:
    data = features.dict()
    df   = pd.DataFrame([data])
    for col in feature_cols:
        if col not in df.columns:
            df[col] = 0
    return df[feature_cols]

# ── Routes ──────────────────────────────────────────────────────────────────
@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict/duration", response_model=DurationResponse)
def predict_duration(features: TripFeatures):
    try:
        X    = to_dataframe(features)
        pred = float(reg_model.predict(X)[0])
        return DurationResponse(
            trip_duration_sec=round(pred, 1),
            trip_duration_min=round(pred / 60, 2)
        )
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

@app.post("/predict/type", response_model=TypeResponse)
def predict_type(features: TripFeatures):
    try:
        X     = to_dataframe(features)
        label = int(cls_model.predict(X)[0])
        proba = cls_model.predict_proba(X)[0].tolist() \
                if hasattr(cls_model, "predict_proba") else None
        return TypeResponse(long_trip=bool(label), label=label, probabilities=proba)
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))
'''

with open('app_fastapi.py', 'w') as f:
    f.write(fastapi_code.strip())

print('app_fastapi.py généré.')
print()
print('Pour lancer l\'API FastAPI :')
print('  pip install fastapi uvicorn')
print('  uvicorn app_fastapi:app --reload --port 8000')
print()
print('Documentation Swagger auto-générée : http://localhost:8000/docs')

**10.4 Interface Utilisateur (Gradio)**

In [ ]:
try:
    import gradio as gr
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gradio", "-q"])
    import gradio as gr

import joblib
import pandas as pd
import numpy as np

# Chargement des artefacts
_reg_model    = joblib.load('model_artifacts/best_reg_model.joblib')
_cls_model    = joblib.load('model_artifacts/best_cls_model.joblib')
_feature_cols = joblib.load('model_artifacts/feature_names.joblib')

def make_df(pickup_hour, distance_km, passenger_count,
             pickup_longitude, pickup_latitude,
             dropoff_longitude, dropoff_latitude):
    row = {
        'pickup_hour'      : pickup_hour,
        'distance_km'      : distance_km,
        'passenger_count'  : passenger_count,
        'pickup_longitude' : pickup_longitude,
        'pickup_latitude'  : pickup_latitude,
        'dropoff_longitude': dropoff_longitude,
        'dropoff_latitude' : dropoff_latitude,
    }
    df = pd.DataFrame([row])
    for col in _feature_cols:
        if col not in df.columns:
            df[col] = 0
    return df[_feature_cols]

def predict_all(pickup_hour, distance_km, passenger_count,
                pickup_longitude, pickup_latitude,
                dropoff_longitude, dropoff_latitude):
    X = make_df(pickup_hour, distance_km, passenger_count,
                pickup_longitude, pickup_latitude,
                dropoff_longitude, dropoff_latitude)

    # Régression
    duration_sec = float(_reg_model.predict(X)[0])
    duration_min = duration_sec / 60

    # Classification
    label = int(_cls_model.predict(X)[0])
    trip_type = 'Long trajet' if label == 1 else 'Trajet court'
    if hasattr(_cls_model, 'predict_proba'):
        proba = _cls_model.predict_proba(X)[0]
        confidence = f'{max(proba)*100:.1f}%'
    else:
        confidence = 'N/A'

    return (
        f'{duration_sec:.0f} sec  ({duration_min:.1f} min)',
        trip_type,
        confidence
    )

with gr.Blocks(title='NYC Taxi Predictor', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🚕 NYC Taxi Trip Predictor')
    gr.Markdown('Entrez les caractéristiques du trajet pour obtenir une prédiction.')

    with gr.Row():
        with gr.Column():
            gr.Markdown('### Paramètres du trajet')
            hour       = gr.Slider(0, 23, value=8,   step=1,   label='Heure de départ')
            dist       = gr.Number(value=3.5,               label='Distance (km)')
            passengers = gr.Slider(1, 6,  value=1,   step=1,   label='Nombre de passagers')

        with gr.Column():
            gr.Markdown('### Coordonnées GPS (optionnel)')
            pu_lon = gr.Number(value=-73.985, label='Longitude départ')
            pu_lat = gr.Number(value=40.758,  label='Latitude départ')
            do_lon = gr.Number(value=-73.960, label='Longitude arrivée')
            do_lat = gr.Number(value=40.780,  label='Latitude arrivée')

    btn = gr.Button('Prédire', variant='primary')

    with gr.Row():
        out_duration = gr.Textbox(label='Durée estimée')
        out_type     = gr.Textbox(label='Type de trajet')
        out_conf     = gr.Textbox(label='Confiance')

    btn.click(
        fn=predict_all,
        inputs=[hour, dist, passengers, pu_lon, pu_lat, do_lon, do_lat],
        outputs=[out_duration, out_type, out_conf]
    )

demo.launch(share=True)  # share=True génère un lien public temporaire

**10.5 Monitoring du Modèle**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import ks_2samp, chi2_contingency
from sklearn.metrics import r2_score, mean_squared_error, f1_score
import warnings
warnings.filterwarnings('ignore')

# ── Simulation de données de production ─────────────────────────────────────
np.random.seed(99)
n_prod = 2000

# Production = distribution légèrement décalée (drift simulé)
X_prod = X_test.sample(n=n_prod, replace=True, random_state=99).copy()
X_prod['pickup_hour'] = (X_prod['pickup_hour'] + np.random.randint(-2, 3, n_prod)) % 24
if 'distance_km' in X_prod.columns:
    X_prod['distance_km'] *= np.random.uniform(0.85, 1.20, n_prod)

y_prod_true = y_test.sample(n=n_prod, replace=True, random_state=99).values
y_prod_pred = best_reg_model.predict(X_prod)

print('Données de production simulées :', X_prod.shape)
print(f'R²  production  : {r2_score(y_prod_true, y_prod_pred):.4f}')
print(f'R²  référence   : {r2_score(y_test, best_reg_model.predict(X_test)):.4f}')

10.5.1 Détection du Data Drift (test KS)

In [ ]:
numerical_features = X_test.select_dtypes(include=np.number).columns.tolist()[:6]

drift_report = []
for col in numerical_features:
    stat, pval = ks_2samp(X_test[col].values, X_prod[col].values)
    drift_detected = pval < 0.05
    drift_report.append({
        'Feature'        : col,
        'KS Statistic'   : round(stat, 4),
        'p-value'        : round(pval, 4),
        'Drift détecté?' : ' OUI' if drift_detected else '  NON'
    })

df_drift = pd.DataFrame(drift_report)
display(df_drift.style
    .applymap(lambda v: 'color: red; font-weight: bold' if '⚠' in str(v) else '',
              subset=['Drift détecté?'])
    .background_gradient(cmap='Oranges', subset=['KS Statistic'])
    .format({'KS Statistic': '{:.4f}', 'p-value': '{:.4f}'}))

10.5.2 Visualisation des distributions Référence vs Production

In [ ]:
n_feat = min(4, len(numerical_features))
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for i, col in enumerate(numerical_features[:n_feat]):
    ax = axes[i]
    ref_vals  = X_test[col].values
    prod_vals = X_prod[col].values

    _, pval = ks_2samp(ref_vals, prod_vals)
    color   = '#e74c3c' if pval < 0.05 else '#2ecc71'

    ax.hist(ref_vals,  bins=40, alpha=0.55, color='#3498db', label='Référence (test)', density=True)
    ax.hist(prod_vals, bins=40, alpha=0.55, color=color,     label='Production',       density=True)
    ax.set_title(f'{col}  (p={pval:.3f})', fontweight='bold',
                 color='#e74c3c' if pval < 0.05 else 'black')
    ax.legend(fontsize=8)
    ax.set_xlabel(col)
    ax.set_ylabel('Densité')

plt.suptitle('Distribution des Features : Référence vs Production', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

10.5.3 Suivi des métriques dans le temps (simulation fenêtre glissante)

In [ ]:
window_size = 200
n_windows   = n_prod // window_size

windows_r2   = []
windows_rmse = []

for w in range(n_windows):
    start = w * window_size
    end   = start + window_size
    y_t   = y_prod_true[start:end]
    y_p   = y_prod_pred[start:end]
    windows_r2.append(r2_score(y_t, y_p))
    windows_rmse.append(np.sqrt(mean_squared_error(y_t, y_p)))

baseline_r2   = r2_score(y_test, best_reg_model.predict(X_test))
baseline_rmse = np.sqrt(mean_squared_error(y_test, best_reg_model.predict(X_test)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(windows_r2,   marker='o', color='#3498db', linewidth=2, markersize=5)
ax1.axhline(baseline_r2, color='red', linestyle='--', linewidth=1.5, label=f'Baseline R²={baseline_r2:.3f}')
ax1.fill_between(range(n_windows),
                 [baseline_r2 * 0.95] * n_windows,
                 [baseline_r2 * 1.05] * n_windows,
                 alpha=0.15, color='red', label='Zone tolérance ±5%')
ax1.set_title('R² — Fenêtres temporelles', fontweight='bold')
ax1.set_xlabel('Fenêtre'); ax1.set_ylabel('R²')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(windows_rmse, marker='s', color='#e67e22', linewidth=2, markersize=5)
ax2.axhline(baseline_rmse, color='red', linestyle='--', linewidth=1.5, label=f'Baseline RMSE={baseline_rmse:.0f}')
ax2.fill_between(range(n_windows),
                 [baseline_rmse * 0.95] * n_windows,
                 [baseline_rmse * 1.05] * n_windows,
                 alpha=0.15, color='red', label='Zone tolérance ±5%')
ax2.set_title('RMSE — Fenêtres temporelles', fontweight='bold')
ax2.set_xlabel('Fenêtre'); ax2.set_ylabel('RMSE (sec)')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('Monitoring des Performances en Production', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Rapport textuel
n_degraded = sum(1 for r in windows_r2 if r < baseline_r2 * 0.95)
print(f'\nBilan monitoring :')
print(f'  Fenêtres analysées            : {n_windows}')
print(f'  Fenêtres en dégradation (R²)  : {n_degraded} / {n_windows}')
if n_degraded > n_windows * 0.3:
    print('ACTION REQUISE : réentraîner le modèle.')
else:
    print('Performances stables.')

**10.6 Synthèse — Checklist de déploiement**

| Étape | Outil | Statut |
|---|---|---|
| Sérialisation du modèle | `joblib` | ✅ |
| API légère (REST) | Flask | ✅ |
| API robuste (validation + docs) | FastAPI | ✅ |
| Interface utilisateur interactive | Gradio | ✅ |
| Détection data drift | Test KS (`scipy`) | ✅ |
| Suivi métriques fenêtre glissante | Matplotlib | ✅ |

